처음 한 번만 실행

mkdir -p models

python -c "from huggingface_hub import hf_hub_download; hf_hub_download(repo_id='unsloth/gemma-3-27b-it-GGUF', filename='gemma-3-27b-it-UD-Q6_K_XL.gguf', local_dir='./models', local_dir_use_symlinks=False)"

매 번 실행

export LD_LIBRARY_PATH=/usr/local/cuda/lib64:$LD_LIBRARY_PATH

./llama.cpp/build/bin/llama-server \
  -m ./models/gemma-3-27b-it-UD-Q6_K_XL.gguf \
  -c 20000 \
  -np 4 \
  -cb \
  -fa on \
  --port 8000 \
  --host 0.0.0.0

In [ ]:
import asyncio
import ast
import pandas as pd
from openai import AsyncOpenAI

model_name = "Gemma3-27B"
df_test = pd.read_csv('test.csv', index_col=0)
df_output = pd.read_csv('output.csv')

client = AsyncOpenAI(
    base_url="http://localhost:8000/v1",
    api_key="sk-no-key-required",
    timeout=1500.0,
)

In [ ]:
system_msg_0 = """객관식 문제가 주어집니다. 다른 출력은 절대 포함하지 말고, 오직 이 문제가 속하는 과목을 "과목명 - 단원명" 형식으로만 응답해주세요."""

system_msg_1 = """객관식 문제가 주어집니다. 다른 출력은 절대 포함하지 말고, 오직 이 문제가 어떤 과목의 문제인지를 "과목명 - 단원명" 형식으로만 응답해주세요."""

system_msg_2 = """객관식 문제가 주어집니다. 다른 출력은 절대 포함하지 말고, 오직 이 문제와 관련된 지식분야를 "대분류 - 세부분류" 형식으로만 응답해주세요."""

system_msg_3 = """객관식 문제가 주어집니다. 다른 출력은 절대 포함하지 말고, 오직 이 문제가 어떤 지식영역의 문제인지를 "대분류 - 세부분류" 형식으로만 응답해주세요."""


list_system_msg = [system_msg_0, system_msg_1, system_msg_2, system_msg_3]

In [ ]:
async def get_inference(system_msg, user_content, seed):
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_msg},
                {"role": "user", "content": user_content}
            ],
            max_tokens=256,
            temperature=1.0,
            top_p=0.95,
            seed=seed,
            extra_body={"min_p": 0.0, "top_k": 30}
        )
        ret = response.choices[0].message.content
        return ret

    except Exception as e:
        print(f"Inference {seed} Error: {e}")
        return "Error"


async def process_row(index, row, seed_base):
    problem = ast.literal_eval(row['problems'])
    user_content = f"<제시문>\n{row['paragraph']}\n\n"
    if pd.notna(row['question_plus']):
        user_content += f"<보기>\n{row['question_plus']}\n\n"
    user_content += f"<질문>\n{problem['question']}\n"
    for k in range(len(problem['choices'])):
        user_content += f"{k+1}. {problem['choices'][k]}\n"

    seed = (seed_base * len(df_test) + index)

    tasks = []
    for r in range(len(list_system_msg)):
        tasks.append(get_inference(list_system_msg[r], user_content, seed))
    
    results = await asyncio.gather(*tasks)
    
    return index, results


async def main():
    print(f"Start Classification with Model: {model_name}")

    for s in range(0, 3):
        print(f"==== Processing Loop s={s} ====")

        for i in range(len(df_test)):
            print(f"Processing s={s}, i={i} (Parallel requests for {len(list_system_msg)} personas)...")
            
            idx, results = await process_row(i, df_test.loc[i], s)
            
            for r, output in enumerate(results):
                df_test.loc[idx, f'class_{r}_{s}'] = output.strip()
            
            df_test.to_csv(f'TestSet_Classification_{model_name}.csv', index=False)
        df_test.to_csv(f'TestSet_Classification_{model_name}.csv', index=False)

In [ ]:
await main()